<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
%cd /content
!git clone --depth 1 https://github.com/Fatima-05/FlyRank-ML
%cd FlyRank-ML
df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head(3)

/content
fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.
/content/FlyRank-ML
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# One row=one content page (one content_id)

# Time window: the starter data already gives 90 day aggregates per page
# For ranking pages to refresh, I treat each row as one page summarized over that lookback window

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# Features: impressions_90d, avg_position, ctr, days_since_last_update/content_age_days, word_count, sessions_90d (and simple logs/tiers derived only from these).

# Label/proxy: is_declining_label=(trend_direction=="down"). This is a same-window proxy, not a pure future outcome.

# Context only: content_id, client_id (for grouping/holdout, not as meaningful features).

# Excluded:
# 1) Any product scores or action flags (not present, and would be circular).
# 2) Raw URLs, titles, client names, queries (not present and must stay out).
# 3) Any information from after the decision moment (leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Query 1: Grain
print("Rows:",len(df))
print("Unique content_id:",df["content_id"].nunique())
print("One row = one page?",len(df)==df["content_id"].nunique())

# Query 2: Size+basic span
print("\nDeclining rate:",(df["trend_direction"].str.lower()=="down").mean().round(3))
print("Impressions > 0:",(df["impressions_90d"]>0).sum())

# Query 3: Availability style filter
usable=df[(df["impressions_90d"] > 0) & (df["content_age_days"]>=90)]
print("\nRows after filter (impressions>0 & age>=90):",len(usable))
print("Fraction kept:",round(len(usable)/len(df),3))

Rows: 30000
Unique content_id: 30000
One row = one page? True

Declining rate: 0.542
Impressions > 0: 30000

Rows after filter (impressions>0 & age>=90): 30000
Fraction kept: 1.0


| Feature | Knowable at decision time because… |
|---------|------------------------------------|
| log(impressions_90d) | Past 90-day observations only |
| avg_position | Observed in the feature window |
| content_age_days | Known page metadata |
| ctr | Clicks/impressions from the past window |
| word_count | Known page attribute |

In [7]:
# Deliberate leak
df["leak_from_label"]=(df["trend_direction"].str.lower()=="down").astype(int)

print("Correlation of leak feature with label:",
      df["leak_from_label"].corr(df["trend_direction"].str.lower().eq("down").astype(int)))

# Removing the leak
df=df.drop(columns=["leak_from_label"])
print("Leak feature deleted. Only honest features remain.")

Correlation of leak feature with label: 1.0
Leak feature deleted. Only honest features remain.


When the label derived column was added, the relationship looked perfect. That score was fake. After removing it, only the honest features remain.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# Named limitation:
# The current label (trend_direction=="down") is a same window proxy. It helps learn the ranking workflow, but it does not prove that refreshing a page caused recovery. A stronger contract needs a clear past feature window and a later outcome window as well as a client level holdout

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.